# Personal Writer's Assistant

1. Analyses a snippet of past work and forces a **Structured Output**.
2. Passes the style rules to a **Drafter Agent** to write on a new topic in **British English**.
3. **Hands off** the draft to a strict **Editor Agent** to refine the style.

In [ ]:
from agents import Agent, Runner, trace
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(override=True)

### Structured Output for Style Analysis

In [ ]:
class StyleGuide(BaseModel):
    tone: str = Field(description="The overall tone of the writing (e.g., witty, serious, conversational).")
    sentence_structure: str = Field(description="Observations on sentence length and complexity.")
    vocabulary: str = Field(description="Type of words used (e.g., simple, academic, colourful).")

analyser_agent = Agent(
    name="Style Analyser",
    instructions="You analyse a piece of writing and extract its style characteristics. Make sure to note British English markers if present.",
    model="gpt-4o-mini",
    output_type=StyleGuide
)

In [ ]:
past_work = "Life is entirely too short to drink bad coffee or write boring code. Let's build things that matter, and maybe have a laugh while doing it. We should embrace the colourful array of possibilities in software engineering."

with trace("Analyse Style"):
    style_result = await Runner.run(analyser_agent, past_work)

style_guide = style_result.final_output
print("Extracted Style Guide:\n", style_guide.model_dump_json(indent=2))

The Drafter takes the style guide directly in its prompt. It is strictly instructed to use British English. Once done, it is instructed to hand off control to an Editor Agent, proving Agent-to-Agent collaboration.

In [ ]:
editor_agent = Agent(
    name="Harsh Editor",
    instructions="You receive a draft. Your job is to make the draft sound more human and remove generic filler words like 'In conclusion', 'today', or 'delve'. Ensure strict use of British English spelling (e.g., 'colour', 'analyse') and grammar. Output only the final edited text.",
    model="gpt-4o-mini"
)

drafter_agent = Agent(
    name="Style Drafter",
    instructions="You are a writer. You will be given a Topic and a Style Guide in JSON format. Write a short 2-sentence paragraph about the Topic using EXACTLY the rules in the Style Guide. Always use British English spelling and grammar. After drafting, hand off to the 'Harsh Editor' to review it.",
    model="gpt-4o-mini",
    handoffs=[editor_agent]
)

In [ ]:
new_topic = "The importance of taking proper breaks to re-energise while programming."
prompt = f"Topic: {new_topic}\nStyle Guide: {style_guide.model_dump_json()}"

with trace("Draft and Edit Handoff"):
    final_result = await Runner.run(drafter_agent, prompt)

print("Final Edited Draft:\n", final_result.final_output)